[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/S08_06_leaderboard_plantilla.ipynb)

# S08 · 06 — Plantilla del *leaderboard* interno («Duelo de modelos»)

**Tiempo estimado:** 45 min (herramienta del reto PBL del Sprint 8)

**Contexto TurisData.** En el reto «Duelo de modelos» vuestro equipo compara árboles, bosques, boosting y SVM
para predecir cancelaciones y debe **elegir un modelo campeón** justificando **rendimiento, explicabilidad y
coste**. Este notebook es la **plantilla del leaderboard**: una tabla que se va rellenando (y se guarda en un
CSV) con la misma medición para todos los modelos, para que la comparación sea justa y reproducible.

**Cómo se usa:**
1. La celda de *herramientas* ya está hecha: **no la cambies** (así todos los equipos miden igual).
2. Registra tus modelos con `registrar(evaluar_modelo(...))`.
3. Calcula la puntuación global con **tus pesos** (y justifícalos en la *model card*).
4. Evalúa **solo al campeón** en el test, una vez.

**Al terminar tendrás:** `leaderboard_S08.csv` (la tabla) y los datos para rellenar la *model card*
(`guias/S08/model_card_plantilla.md`).

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os
import pickle
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
SEMILLA = 42
sns.set_theme(style="whitegrid")

RUTA_CSV = "leaderboard_S08.csv"     # el leaderboard se guarda aquí (en la carpeta del notebook)
REINICIAR = True                      # True = empieza de cero al ejecutar todo; False = conserva lo registrado

## 1. Datos y partición

Mismo problema que en las prácticas: predecir `cancelada`. El **test se reserva** hasta el final; todas las
comparaciones del leaderboard se hacen con **validación cruzada sobre train** (5 folds).

In [ ]:
df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)
print(f"Train: {len(X_train)} · Test (intacto hasta el final): {len(X_test)}")

## 2. Herramientas del leaderboard (no modificar)

- `hacer_pipe(clasificador, escalar)`: preprocesado estándar + el clasificador que le pases.
- `evaluar_modelo(...)`: mide **AUC y AP (validación cruzada)**, **tiempo de ajuste**, **tiempo de predicción
  por cada 1000 reservas** y **tamaño del modelo** en KB. La **explicabilidad** (1 = caja negra, 5 = se lee
  directamente) la pones tú, con criterio.
- `registrar(resultado)`: añade (o actualiza) una fila en el CSV y devuelve el leaderboard completo.
- `puntuacion_global(lb, pesos)`: combina rendimiento, explicabilidad y coste (los tres normalizados entre 0 y 1).

In [ ]:
def hacer_pipe(clasificador, escalar=False):
    """Preprocesado estándar (imputar + one-hot; escalar si el modelo lo necesita) + clasificador."""
    num = [("imp", SimpleImputer(strategy="median"))] + ([("esc", StandardScaler())] if escalar else [])
    prep = ColumnTransformer([
        ("num", Pipeline(num), columnas_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
    ])
    return Pipeline([("prep", prep), ("clf", clasificador)])


def evaluar_modelo(nombre, familia, pipe, explicabilidad, notas=""):
    """Mide un modelo con validación cruzada sobre train y devuelve una fila del leaderboard."""
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision"], n_jobs=1)
    pipe.fit(X_train, y_train)
    muestra = X_train.head(1000)
    t0 = time.perf_counter()
    pipe.predict_proba(muestra)
    ms_pred = (time.perf_counter() - t0) * 1000 * (1000 / len(muestra))
    return {
        "modelo": nombre, "familia": familia,
        "auc_cv_media": round(res["test_roc_auc"].mean(), 4), "auc_cv_desv": round(res["test_roc_auc"].std(), 4),
        "ap_cv_media": round(res["test_average_precision"].mean(), 4),
        "seg_ajuste": round(res["fit_time"].mean(), 3), "ms_pred_1000": round(ms_pred, 1),
        "tamano_kb": round(len(pickle.dumps(pipe)) / 1024, 1),
        "explicabilidad_1a5": explicabilidad, "notas": notas, "auc_test": np.nan,
    }


def registrar(fila):
    """Añade o actualiza una fila (por nombre de modelo) en el CSV y devuelve la tabla ordenada por AUC."""
    tabla = pd.read_csv(RUTA_CSV) if os.path.exists(RUTA_CSV) else pd.DataFrame()
    if len(tabla) and fila["modelo"] in set(tabla["modelo"]):
        conservar = tabla.loc[tabla["modelo"] == fila["modelo"], "auc_test"].iloc[0]
        fila = {**fila, "auc_test": conservar}
        tabla = tabla[tabla["modelo"] != fila["modelo"]]
    tabla = pd.concat([tabla, pd.DataFrame([fila])], ignore_index=True)
    tabla = tabla.sort_values("auc_cv_media", ascending=False).reset_index(drop=True)
    tabla.to_csv(RUTA_CSV, index=False)
    return tabla


def puntuacion_global(lb, pesos):
    """Puntuación 0-1 = rendimiento (AUC), explicabilidad y coste (tiempo + tamaño), normalizados y ponderados."""
    rend = (lb["auc_cv_media"] - lb["auc_cv_media"].min()) / (lb["auc_cv_media"].max() - lb["auc_cv_media"].min() + 1e-9)
    expl = (lb["explicabilidad_1a5"] - 1) / 4
    coste_bruto = np.log1p(lb["seg_ajuste"]) + np.log1p(lb["ms_pred_1000"]) + np.log1p(lb["tamano_kb"])
    coste = 1 - (coste_bruto - coste_bruto.min()) / (coste_bruto.max() - coste_bruto.min() + 1e-9)   # 1 = barato
    total = pesos["rendimiento"] * rend + pesos["explicabilidad"] * expl + pesos["coste"] * coste
    return lb.assign(punt_rendimiento=rend.round(3), punt_explicabilidad=expl.round(3),
                     punt_coste=coste.round(3), puntuacion_global=total.round(3)).sort_values("puntuacion_global", ascending=False)


if REINICIAR and os.path.exists(RUTA_CSV):
    os.remove(RUTA_CSV)
print("Herramientas listas. Leaderboard en:", RUTA_CSV)

## 3. Registrar el baseline y el modelo lineal

**Ejercicio 1.** Registra dos modelos con `registrar(evaluar_modelo(...))` y guarda el resultado de la última
llamada en `lb`:
- `"baseline"` (familia `"referencia"`): `hacer_pipe(DummyClassifier(strategy="prior"))`, explicabilidad 5,
- `"logistica"` (familia `"lineal"`): `hacer_pipe(LogisticRegression(max_iter=1000), escalar=True)`, explicabilidad 4.

In [ ]:
# TODO: registrar(evaluar_modelo(nombre, familia, hacer_pipe(...), explicabilidad, notas)) para cada modelo
...
assert len(lb) == 2 and set(lb["modelo"]) == {"baseline", "logistica"}, "El leaderboard debe tener baseline y logistica"
assert abs(lb.set_index("modelo").loc["baseline", "auc_cv_media"] - 0.5) < 0.01, "El baseline debe dar AUC ≈ 0,5"
lb

## 4. Registrar los contendientes del duelo

**Ejercicio 2.** Registra estos cuatro modelos (con explicabilidad entre paréntesis) y guarda el leaderboard
devuelto en `lb`:
- `"arbol_podado"` (familia `"arbol"`): `DecisionTreeClassifier(min_samples_leaf=80, random_state=SEMILLA)` (5),
- `"random_forest"` (familia `"ensemble_bagging"`): `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)` (2),
- `"lightgbm"` (familia `"ensemble_boosting"`): `lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=8, random_state=SEMILLA, n_jobs=2, verbose=-1)` (2),
- `"svm_rbf"` (familia `"svm"`): `SVC(kernel="rbf", C=1, probability=True, random_state=SEMILLA)` con `escalar=True` (1).

Añade también, si quieres, `"xgboost"` (familia `"ensemble_boosting"`). ¡Los tuyos pueden llevar los hiperparámetros que hayas encontrado con la búsqueda!

In [ ]:
# TODO: mismo patrón que el ejercicio 1; la SVM necesita escalar=True
...
assert len(lb) >= 6, "Deben estar el baseline, la logística y al menos cuatro modelos más"
assert {"arbol_podado", "random_forest", "svm_rbf"} <= set(lb["modelo"]), "Faltan modelos de las familias árbol, bosque o SVM"
assert lb["familia"].str.contains("boosting").any(), "Falta un modelo de boosting"
lb

## 5. Rendimiento frente a explicabilidad y coste

Un gráfico ayuda a ver los **compromisos**: AUC frente al tiempo de ajuste, con el tamaño del punto
proporcional al tamaño del modelo.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.scatter(lb["seg_ajuste"], lb["auc_cv_media"], s=lb["tamano_kb"].clip(lower=5) * 1.2, alpha=0.6, color="#2a6f97")
for _, f in lb.iterrows():
    plt.annotate(f["modelo"], (f["seg_ajuste"], f["auc_cv_media"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
plt.xscale("log")
plt.title("Leaderboard: rendimiento frente a tiempo de ajuste (tamaño del punto = tamaño del modelo)")
plt.xlabel("Segundos de ajuste por fold (escala logarítmica)")
plt.ylabel("AUC (validación cruzada)")
plt.show()

## 6. Puntuación global con **tus** pesos

**Ejercicio 3.** Define los `pesos` (diccionario con las claves `"rendimiento"`, `"explicabilidad"` y `"coste"`,
que **sumen 1**) que reflejen lo que el cliente valora (por ejemplo, un hotel pequeño que quiere entender el modelo
puede dar más peso a la explicabilidad). Calcula `lb_pond = puntuacion_global(lb, pesos)` y guarda en `campeon`
el nombre del modelo con mayor puntuación.

In [ ]:
# TODO: pesos = {"rendimiento": ..., "explicabilidad": ..., "coste": ...}; el campeón es la primera fila
...
assert abs(sum(pesos.values()) - 1) < 1e-9, "Los pesos deben sumar 1"
assert set(pesos) == {"rendimiento", "explicabilidad", "coste"}, "Claves: rendimiento, explicabilidad y coste"
assert campeon in set(lb["modelo"]) and campeon != "baseline", "El campeón debe ser uno de los modelos y no el baseline"
print("Campeón con estos pesos:", campeon)
lb_pond[["modelo", "auc_cv_media", "explicabilidad_1a5", "punt_rendimiento", "punt_explicabilidad", "punt_coste", "puntuacion_global"]]

> **Análisis de sensibilidad.** ¿Cambia el campeón si cambian los pesos? Prueba tres conjuntos de pesos
> (por ejemplo `0.8/0.1/0.1`, `0.4/0.4/0.2` y `0.2/0.6/0.2`). Si el campeón cambia, **tu recomendación depende de
> lo que valore el cliente**, y así debes contarlo en la *model card*.

In [ ]:
escenarios = {"solo rendimiento": {"rendimiento": 0.8, "explicabilidad": 0.1, "coste": 0.1},
              "equilibrado": {"rendimiento": 0.4, "explicabilidad": 0.4, "coste": 0.2},
              "explicabilidad primero": {"rendimiento": 0.2, "explicabilidad": 0.6, "coste": 0.2}}
pd.DataFrame({nombre: [puntuacion_global(lb, p).iloc[0]["modelo"]] for nombre, p in escenarios.items()},
             index=["campeón"]).T

## 7. El examen final: solo el campeón, una vez

**Ejercicio 4.** Reconstruye el campeón con el diccionario `catalogo` (justo debajo: cada entrada crea el
pipeline de un modelo), ajústalo con **todo `X_train`** y calcula `auc_test_campeon`: su AUC en `X_test`.
Guarda el resultado en el CSV con `actualizar_test(campeon, auc_test_campeon)` y el leaderboard resultante
en `lb_final`. (Si has cambiado hiperparámetros al registrar tus modelos, actualiza también `catalogo`.)

In [ ]:
def actualizar_test(nombre, valor):
    """Guarda el AUC de test del campeón en el leaderboard (solo debe llamarse UNA vez)."""
    tabla = pd.read_csv(RUTA_CSV)
    tabla.loc[tabla["modelo"] == nombre, "auc_test"] = round(float(valor), 4)
    tabla.to_csv(RUTA_CSV, index=False)
    return tabla

catalogo = {  # cómo reconstruir cada modelo (ajusta si has cambiado hiperparámetros)
    "logistica": lambda: hacer_pipe(LogisticRegression(max_iter=1000), escalar=True),
    "arbol_podado": lambda: hacer_pipe(DecisionTreeClassifier(min_samples_leaf=80, random_state=SEMILLA)),
    "random_forest": lambda: hacer_pipe(RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)),
    "lightgbm": lambda: hacer_pipe(lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=8, random_state=SEMILLA, n_jobs=2, verbose=-1)),
    "xgboost": lambda: hacer_pipe(xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=SEMILLA, n_jobs=2)),
    "svm_rbf": lambda: hacer_pipe(SVC(kernel="rbf", C=1, probability=True, random_state=SEMILLA), escalar=True),
}
# TODO: catalogo[campeon]() reconstruye el pipeline; ajusta con train; roc_auc_score con X_test
...
assert 0.65 < auc_test_campeon < 0.85, "El AUC de test del campeón debería estar entre 0,65 y 0,85"
assert lb_final["auc_test"].notna().sum() == 1, "Solo el campeón debe tener AUC de test"
print(f"Campeón: {campeon} · AUC en validación cruzada {lb_pond.iloc[0]['auc_cv_media']:.3f} · AUC en test {auc_test_campeon:.3f}")
lb_final

## 8. Y ahora, a la *model card*

Con esta tabla ya tienes los números para rellenar `guias/S08/model_card_plantilla.md`: qué modelo
es, para qué sirve, con qué datos se entrenó, cómo rinde, **qué no debe usarse** y qué riesgos tiene.
Entrega también el CSV `leaderboard_S08.csv`.

## 9. Para reflexionar

1. ¿Por qué medimos todos los modelos con los **mismos folds** y las mismas variables?
2. Si el campeón por rendimiento y el campeón por puntuación global son distintos, ¿a quién y cómo se lo explicas?
3. ¿Qué información falta en el leaderboard que el cliente querría saber (coste de mantenimiento, cómo se actualiza el modelo, riesgo de sesgo…)?
4. Si el test se hubiera usado para elegir, ¿qué habría pasado con la nota del campeón?

## 10. Resumen: qué has aprendido
- Un leaderboard justo mide **igual** todos los modelos (mismos datos, mismos folds, mismas métricas).
- Elegir campeón es una decisión **multicriterio** (rendimiento, explicabilidad, coste); los pesos dependen del cliente.
- El test se usa **una vez**, solo con el campeón.